# 14 - Overlay: BUY/SELL signals on price + how each was decided (AUTO)

Auto-picks the most-signalled symbols. For each it draws the (smooth, weekly)
price line with BUY/SELL markers AND prints the reasoning behind every signal.

**How a signal is decided** (notebook 10): a conviction score 0-5 from a
*conjunctive* gate - an attention take-off (`att_z`) AND a sentiment change
(`sent_5d_chg`) must agree, combined into `conv_z`. BUY needs score >= 4,
SELL >= 3. The `reason` column spells out each factor for that specific call.

In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
P = os.path.join(ROOT, 'data', 'processed')
PRICES_PATH = os.path.join(ROOT, 'data', 'prices', 'prices.parquet')

# window comes from update_data.py (same toggle as live vs backtest)
from update_data import START_DATE, END_DATE
WIN_LO = pd.to_datetime(START_DATE)
WIN_HI = pd.to_datetime(END_DATE) if END_DATE else None
print('window:', WIN_LO.date(), '->', (WIN_HI.date() if WIN_HI is not None else 'LIVE (newest)'))

def clip_series(s):
    s = s[s.index >= WIN_LO]
    return s if WIN_HI is None else s[s.index <= WIN_HI]

def clip_dates(df, col):
    df = df[df[col] >= WIN_LO]
    return df if WIN_HI is None else df[df[col] <= WIN_HI]

def load_prices():
    if not os.path.exists(PRICES_PATH):
        raise FileNotFoundError('prices.parquet not found - run  python pull_bloomberg_prices.py  first.')
    px = pd.read_parquet(PRICES_PATH); px['date'] = pd.to_datetime(px['date'])
    return px

def price_series(prices, symbol):
    # daily close, then made CONTINUOUS (forward-fill weekends/holidays) so the
    # line is smooth with no gaps. Clip to the window.
    one = prices[prices['symbol'] == symbol].sort_values('date')
    s = one.set_index('date')['px_last']
    if not s.empty:
        s = s.asfreq('D').ffill()
    return clip_series(s)


In [ ]:
HOW_MANY = 6
FREQ = 'W'        # 'W' weekly (smooth price line), 'D' daily, 'M' monthly
pd.set_option('display.max_colwidth', None); pd.set_option('display.width', 200)

In [ ]:
tick = pd.read_parquet(os.path.join(P, 'trade_signals_tickers.parquet'))
theme = pd.read_parquet(os.path.join(P, 'trade_signals.parquet'))
tick['action_date'] = pd.to_datetime(tick['action_date']); theme['action_date'] = pd.to_datetime(theme['action_date'])
tick = clip_dates(tick, 'action_date'); theme = clip_dates(theme, 'action_date')

# unify both signal files, carrying the decision detail
DET = ['action_date', 'action', 'score', 'att_z', 'conv_z', 'sent_5d_chg', 'reason']
tk = tick.rename(columns={'ticker': 'symbol'})[['symbol'] + DET]
th = theme.rename(columns={'etf': 'symbol'})[['symbol'] + DET]
sig_all = pd.concat([tk, th], ignore_index=True)
symbols = sig_all['symbol'].value_counts().head(HOW_MANY).index.tolist()
print('auto signalled symbols:', symbols)

prices = load_prices()
for symbol in symbols:
    px_daily = price_series(prices, symbol)
    if px_daily.empty:
        print('skip', symbol, '- no price rows'); continue
    s = sig_all[sig_all['symbol'] == symbol].sort_values('action_date').copy()
    s['price'] = s['action_date'].map(px_daily)     # marker height from the daily price

    # --- the decision detail for this symbol ---
    print('\n' + '=' * 90)
    print(f'{symbol}: {len(s)} signals - how each was decided')
    print('=' * 90)
    print(s[DET].to_string(index=False))

    # --- the chart ---
    px_line = px_daily if FREQ == 'D' else px_daily.resample(FREQ).last()
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(px_line.index, px_line.values, color='black', linewidth=1.3, label='price')
    buys = s[s['action'] == 'BUY']; sells = s[s['action'] == 'SELL']
    ax.scatter(buys['action_date'], buys['price'], marker='^', s=110, color='tab:green', zorder=5, label='BUY')
    ax.scatter(sells['action_date'], sells['price'], marker='v', s=110, color='tab:red', zorder=5, label='SELL')
    # annotate each marker with its score so the chart shows conviction strength
    for _, r in s.iterrows():
        if pd.notna(r['price']):
            ax.annotate(f"{r['action'][0]}{int(r['score'])}", (r['action_date'], r['price']),
                        textcoords='offset points', xytext=(0, 8), fontsize=8, ha='center')
    ax.set_ylabel('price (USD)'); ax.set_title(f'{symbol}: price with BUY/SELL signals ({FREQ})')
    ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout(); plt.show()